<a href="https://colab.research.google.com/github/1900690/OCR/blob/main/OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 OCR表抽出アプリ 使い方ガイド

このプログラムは、PDFや画像ファイルから表を自動で検出し、CSV形式のデータとして抽出するツールです。Google Colab上で動作し、複数ページの解析や結果の自動ダウンロードに対応しています。
<table style="border: none; border-collapse: collapse;">
  <tr style="border: none;">
    <td style="border: none; padding: 10px; vertical-align: top;">
      <img src="https://github.com/1900690/OCR/raw/main/text/setumei1.png" width="800">
      <div style="text-align: center;">文字を認識したもの</div>
    </td>
    <td style="border: none; padding: 10px; vertical-align: top;">
      <img src="https://github.com/1900690/OCR/raw/main/text/setumei2.png" width="1500">
      <div style="text-align: center;">出力されるCSV</div>
    </td>
  </tr>
</table>

---
### 1. 画像の準備
*   分析を画像で行う場合は、影ができないように撮影してください
*   スキャンしたPDFで解析を行う場合は、スキャン解像度を最大にしてください
---

### 2. 操作手順
1.  **ランタイムの確認**: 上部メニューの「ランタイム」→「ランタイムのタイプを変更」で、T4 GPUが選択されていることを確認してください。
2.  **事前準備**: このセルをはじめに実行してくださいと書いてあるセルの左側にある実行ボタン（▶️）をクリックします。
3.  **設定の調整**: OCR実行セルのフォームで、ファイルの取得方法やダウンロードの有無を選択します。
4.  **セルの実行**: セルの左側にある実行ボタン（▶️）をクリックします。
5.  **ファイルの選択**: `SOURCE_TYPE` で「Upload File」を選んだ場合、ファイル選択ボタンが表示されるので、対象ファイル（PDFまたは画像）を選んでください。
6.  **解析と結果表示**: 解析が終わると、画面上に**プレビュー画像**が表示されます。
7.  **ファイルの取得**:
    *   ダウンロード設定が有効な場合、ブラウザ経由でCSVや画像が保存されます。
    *   ※ブラウザが「複数ファイルのダウンロード」をブロックした場合は、アドレスバー右端のアイコンから許可してください。

---

In [ ]:
#@title 事前準備
#@markdown　このセルをはじめに実行してください
# パッケージのインストール
!pip install yomitoku nest_asyncio pandas lxml -q

import nest_asyncio
nest_asyncio.apply()

import os
import cv2
import io
import numpy as np
import pandas as pd
from google.colab import files
from google.colab.patches import cv2_imshow
from yomitoku import DocumentAnalyzer
from yomitoku.data.functions import load_pdf, load_image

# モデルの初期化
analyzer = DocumentAnalyzer(visualize=True, device="cuda")

print("★事前準備完了★")

In [ ]:
# @title 📄 OCR実行

# --- フォーム設定項目 ---
# @markdown ### 1. 解析・出力設定
# @markdown 上部の切取り部分を設定してください
TOP_CROP_RATIO = 0.06 # @param {type:"number"}
DISPLAY_WIDTH = 800

# @markdown ### 2. ファイル取得方法
# @markdown ”サンプルで分析を行う”または”アップロードしたファイルで分析を行う”のどちらかを選択してください
SOURCE_TYPE = "Use Sample" # @param ["Use Sample", "Upload File"]
SAMPLE_URL = "https://github.com/1900690/OCR/raw/main/SKM_C301i26051411120.pdf"

# @markdown ### 3. ダウンロード設定
# @markdown チェックを入れることで分析結果のCSV、分析結果の画像をダウンロードすることができます。

DOWNLOAD_CSV = False # @param {type:"boolean"}
DOWNLOAD_IMAGE = False # @param {type:"boolean"}

# ---------------------------------------------------------
import os
import io
import logging
import cv2
import numpy as np
import pandas as pd
from google.colab import files
from google.colab.patches import cv2_imshow

def setup_quiet_mode():
    """ライブラリの冗長なログを徹底的に非表示にする"""
    for logger_name in logging.root.manager.loggerDict:
        if "yomitoku" in logger_name:
            logger = logging.getLogger(logger_name)
            logger.setLevel(logging.ERROR)
            for handler in logger.handlers[:]:
                logger.removeHandler(handler)
            logger.propagate = False

    logging.getLogger("yomitoku").setLevel(logging.ERROR)
    logging.basicConfig(level=logging.WARNING, force=True)

def get_input_file(source_type, sample_url):
    """ファイルのアップロードまたはサンプル取得"""
    if source_type == "Use Sample":
        path = "sample_data.pdf"
        print(" サンプルファイルをダウンロード中...")
        !wget -q {sample_url} -O {path}
        return path
    else:
        print(" 解析したいファイル（PDF/画像）を選択してください。")
        uploaded = files.upload()
        return list(uploaded.keys())[0] if uploaded else None

def process_single_page(img, page_idx):
    """1ページ分の解析・保存・表示・ダウンロード処理を行う"""
    page_num = page_idx + 1
    print(f"\n---  ページ {page_num} の解析を開始 ---")

    # 1. 前処理（numpy変換 & トリミング）
    img_arr = np.array(img) if not isinstance(img, np.ndarray) else img
    h, w = img_arr.shape[:2]
    processed_img = img_arr[int(h * TOP_CROP_RATIO):h, 0:w]

    # 2. OCR解析実行
    # ※ analyzer, load_pdf, load_image は事前に定義されている前提です
    results, ocr_vis, _ = analyzer(processed_img)

    # 3. CSV変換と保存
    # 修正点：to_html に保存用パスを渡すようにしました
    out_html_path = f"result_p{page_num}.html"
    html_content = results.to_html(out_path=out_html_path, img=processed_img)

    try:
        tables = pd.read_html(io.StringIO(html_content), header=0)
        if tables:
            for t_idx, df in enumerate(tables):
                csv_path = f"result_p{page_num}_table_{t_idx}.csv"
                df.to_csv(csv_path, index=False, encoding="utf-8-sig", float_format='%.0f')
                print(f" CSV作成完了: {csv_path}")
                if DOWNLOAD_CSV:
                    files.download(csv_path)
        else:
            print(f" ページ {page_num} に表は見つかりませんでした。")
    except Exception as e:
        print(f" CSV変換エラー: {e}")

    # 4. 可視化結果の表示
    vis_h, vis_w = ocr_vis.shape[:2]
    target_h = int(DISPLAY_WIDTH * (vis_h / vis_w))
    resized_vis = cv2.resize(ocr_vis, (DISPLAY_WIDTH, target_h))

    print(f"【ページ {page_num} OCR検出結果】")
    cv2_imshow(resized_vis)

    # 5. 画像保存とダウンロード
    img_filename = f"result_ocr_p{page_num}.jpg"
    cv2.imwrite(img_filename, ocr_vis)
    if DOWNLOAD_IMAGE:
        print(f" 画像をダウンロードします: {img_filename}")
        files.download(img_filename)

def main():
    setup_quiet_mode()

    file_path = get_input_file(SOURCE_TYPE, SAMPLE_URL)
    if not file_path:
        print(" ファイルが選択されませんでした。")
        return

    # ファイル読み込み
    ext = os.path.splitext(file_path)[1].lower()
    print(f" ファイルを読み込んでいます: {file_path}")

    if ext == '.pdf':
        imgs = load_pdf(file_path)
    elif ext in ['.png', '.jpg', '.jpeg']:
        imgs = load_image(file_path)
    else:
        print(f"未対応のファイル形式です: {ext}")
        return

    # 全ページをループ処理
    for idx, img in enumerate(imgs):
        try:
            process_single_page(img, idx)
        except Exception as e:
            # ここで発生したエラー（今回のような引数不足など）を詳細に表示します
            import traceback
            print(f" ページ {idx+1} の処理中にエラーが発生しました:")
            print(traceback.format_exc())

    print("\n すべての処理が完了しました。")

# 実行
if __name__ == "__main__":
    main()